# Lab 1: Find-S Algorithm
**Dataset:** Car Evaluation (Kaggle / UCI)
**Goal:** Find the most specific hypothesis that describes a *very good* (`vgood`) car.

## 1. Find-S Algorithm (Theory)
- Start with the most specific hypothesis: `h = <0, 0, 0, 0, 0, 0>`
- Use **only positive examples**, ignore negative ones
- First positive example: copy it into `h`
- Next positive examples: if an attribute value differs from `h`, replace it with `?`
- Final `h` = most specific hypothesis that covers all positives

**Limitations:** ignores negatives, needs noise-free data, gives only one hypothesis.

## 2. Dataset Description
| Attribute | Meaning | Values |
|---|---|---|
| buying | Buying price | vhigh, high, med, low |
| maint | Maintenance cost | vhigh, high, med, low |
| doors | Number of doors | 2, 3, 4, 5more |
| persons | Seating capacity | 2, 4, more |
| lug_boot | Boot size | small, med, big |
| safety | Safety rating | low, med, high |
| class | Car quality | unacc, acc, good, vgood |

**Positive example** = `vgood` car. **Negative example** = any other class.

Download the CSV from Kaggle ("Car Evaluation Data Set") and save it as `data/car_evaluation.csv` (already included in this repo).

## 3. Import Libraries and Load Data

In [1]:
import pandas as pd

COLS = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]

df = pd.read_csv("data/car_evaluation.csv", header=None)
if str(df.iloc[0, 0]).strip().lower() == "buying":   # file already has a header row
    df = pd.read_csv("data/car_evaluation.csv")
df.columns = COLS
df = df.astype(str).apply(lambda s: s.str.strip())

print("Shape:", df.shape)
df.head()

Shape: (1728, 7)


,buying,maint,doors,persons,lug_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


## 4. Explore the Data

In [2]:
print("Missing values:", df.isnull().sum().sum())
print()
print(df["class"].value_counts())

Missing values: 0

class
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64


## 5. Define Positive Class and Split Data

In [3]:
TARGET = "class"
POSITIVE = "vgood"
attributes = [c for c in df.columns if c != TARGET]

train = df.sample(frac=0.8, random_state=42)
test = df.drop(train.index)

print("Train:", train.shape, "| Test:", test.shape)
print("Positive examples in train:", (train[TARGET] == POSITIVE).sum())

Train: (1382, 7) | Test: (346, 7)
Positive examples in train: 50


## 6. Find-S Implementation (with step-by-step trace)

In [4]:
def find_s(data, trace_steps=10):
    h = None                                  # None = most specific <0,0,0,0,0,0>
    n_pos = 0
    for _, row in data.iterrows():
        if row[TARGET] != POSITIVE:           # ignore negative examples
            continue
        n_pos += 1
        x = row[attributes].tolist()
        if h is None:                         # first positive -> copy it
            h = x[:]
        else:                                 # generalize mismatches to '?'
            h = [hv if hv == xv else "?" for hv, xv in zip(h, x)]
        if n_pos <= trace_steps:
            print(f"Positive #{n_pos}: x = {x}")
            print(f"             h = {h}\n")
    print(f"Total positive examples used: {n_pos}")
    return h

h = find_s(train)

Positive #1: x = ['low', 'high', '4', 'more', 'med', 'high']
             h = ['low', 'high', '4', 'more', 'med', 'high']

Positive #2: x = ['low', 'high', '3', 'more', 'med', 'high']
             h = ['low', 'high', '?', 'more', 'med', 'high']

Positive #3: x = ['low', 'med', '4', 'more', 'big', 'high']
             h = ['low', '?', '?', 'more', '?', 'high']

Positive #4: x = ['low', 'med', '3', 'more', 'med', 'high']
             h = ['low', '?', '?', 'more', '?', 'high']

Positive #5: x = ['med', 'low', '2', '4', 'big', 'high']
             h = ['?', '?', '?', '?', '?', 'high']

Positive #6: x = ['med', 'low', '4', 'more', 'med', 'high']
             h = ['?', '?', '?', '?', '?', 'high']

Positive #7: x = ['low', 'low', '3', 'more', 'med', 'high']
             h = ['?', '?', '?', '?', '?', 'high']

Positive #8: x = ['med', 'low', '5more', 'more', 'med', 'high']
             h = ['?', '?', '?', '?', '?', 'high']

Positive #9: x = ['low', 'med', '5more', '4', 'big', 'high']
          

## 7. Final Hypothesis

In [5]:
hyp = pd.DataFrame({"Attribute": attributes, "Value": h})
print("Specific attributes:", sum(v != "?" for v in h), "| Generalized (?):", sum(v == "?" for v in h))
hyp

Specific attributes: 1 | Generalized (?): 5


,Attribute,Value
0,buying,?
1,maint,?
2,doors,?
3,persons,?
4,lug_boot,?
5,safety,high


## 8. Prediction and Evaluation on Test Data

In [6]:
def predict(h, row):
    return all(hv == "?" or hv == row[a] for a, hv in zip(attributes, h))

preds = test.apply(lambda r: POSITIVE if predict(h, r) else "other", axis=1)
actual = test[TARGET].where(test[TARGET] == POSITIVE, "other")

tp = ((preds == POSITIVE) & (actual == POSITIVE)).sum()
fn = ((preds == "other") & (actual == POSITIVE)).sum()
fp = ((preds == POSITIVE) & (actual == "other")).sum()
tn = ((preds == "other") & (actual == "other")).sum()

print(f"Accuracy: {(preds == actual).mean():.4f}")
pd.DataFrame([[tp, fn], [fp, tn]],
             index=["Actual vgood", "Actual other"],
             columns=["Pred vgood", "Pred other"])

Accuracy: 0.7341


,Pred vgood,Pred other
Actual vgood,15,0
Actual other,92,239


## 9. Conclusion and Viva Points
- Find-S learned a single most specific hypothesis covering all `vgood` training examples.
- Attributes marked `?` vary across `vgood` cars, so they do not decide the class.
- Attributes with a fixed value are common to every `vgood` car.
- **FN = 0 on training positives** by design, but **FP can occur** because negatives are ignored.
- Find-S fails on noisy data: one wrong positive can generalize `h` too much.
- Alternative: Candidate Elimination uses both positive and negative examples.